# Human-in-the-loop (HITL)

This notebook demonstrates the **Human-in-the-Loop (HITL)** pattern using the Foundry Agent Service with the Responses API.

When an agent decides to call a tool that requires human oversight - such as an irreversible financial transfer - the Responses API returns the tool call as an output item **without executing it**. The caller then routes the call to an approval branch, presents the details to the operator, and either executes the tool or rejects it based on the decision.

**Scenario:** A financial transaction agent with two tools:
- `get_account_balance` - read-only, auto-executed
- `transfer_funds` - irreversible, requires human approval before execution

**What this notebook demonstrates:**
1. Tool routing via an `APPROVAL_REQUIRED_TOOLS` set
2. Approve path - operator approves a transfer request
3. Reject path - operator rejects a transfer request
4. Multi-step scenario - balance check auto-executes, then transfer requires approval

## Prerequisites

1. **Python environment**: Run `uv sync` from the repository root to create the shared `.venv`, then select the `.venv` kernel in VS Code.
2. **`.env` file**: Must be populated by the `05-foundry-project-pattern-setup` labs:
   - `ALPHA_FOUNDRY_PROJECT_ENDPOINT` - Team Alpha project endpoint URL (set by the project spoke deployment)
   - `ALPHA_FOUNDRY_CORE_CONNECTION` - Team Alpha APIM connection name, e.g. `core-alpha` (set by the project spoke deployment)
   - `CHAT_MODEL` - chat model deployment name, e.g. `gpt-4.1-mini` (set by the core gateway deployment)
3. **Azure CLI**: Run `az login` before executing the cells.
4. **Permissions**: Your identity needs **Azure AI Developer** role on the Foundry project.

> **No new infrastructure required.** This lab reuses the Alpha spoke resources already provisioned by the core gateway and project spoke deployments.

## Step 1: Configuration

Load `.env` from the repository root and initialise the `AIProjectClient` and `openai_client`.

In [1]:
import os
import json
import subprocess
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# ── Change this to rename your agent ─────────────────────────────────────────
AGENT_NAME = "payments-approval-agent"
# ─────────────────────────────────────────────────────────────────────────────

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

# Team Alpha (1:1 spoke) - project endpoint and APIM connection (set by the core gateway and project spoke deployments)
endpoint       = os.environ["ALPHA_FOUNDRY_PROJECT_ENDPOINT"]
hub_connection = os.environ["ALPHA_FOUNDRY_CORE_CONNECTION"]  # e.g. "core-alpha"
chat_model     = os.environ["CHAT_MODEL"]                    # e.g. "gpt-4.1-mini"

# Agents reference models as {connection}/{model} - routes through the APIM connection
model_deployment = f"{hub_connection}/{chat_model}"

credential     = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client  = project_client.get_openai_client()

print(f"Endpoint  : {endpoint}")
print(f"Agent name: {AGENT_NAME}")
print(f"Model     : {model_deployment}")

Endpoint  : https://aif-spoke-alpha-c2676f.services.ai.azure.com/api/projects/project-alpha-c2676f
Agent name: payments-approval-agent
Model     : core-alpha/gpt-4.1-mini


## Step 2: Define tools

Define two `FunctionTool` schemas:
- `get_account_balance` - read-only, safe to auto-execute
- `transfer_funds` - irreversible, requires human approval

The `APPROVAL_REQUIRED_TOOLS` set is the routing convention - any tool name in the set will be intercepted for human approval before execution.

In [2]:
from azure.ai.projects.models import FunctionTool

# Tools in this set require human approval before execution
APPROVAL_REQUIRED_TOOLS = {"transfer_funds"}

get_balance_tool = FunctionTool(
    name="get_account_balance",
    description="Get the current balance for a bank account. Safe to execute automatically.",
    parameters={
        "type": "object",
        "properties": {
            "account_id": {"type": "string", "description": "The account ID, e.g. ACC-001"}
        },
        "required": ["account_id"],
    },
)

transfer_tool = FunctionTool(
    name="transfer_funds",
    description="Transfer funds between bank accounts. REQUIRES human approval before execution.",
    parameters={
        "type": "object",
        "properties": {
            "from_account": {"type": "string", "description": "Source account ID"},
            "to_account":   {"type": "string", "description": "Destination account ID"},
            "amount":       {"type": "number", "description": "Amount in USD to transfer"},
        },
        "required": ["from_account", "to_account", "amount"],
    },
)

# --- Mock implementations ---------------------------------------------------

def get_account_balance(account_id: str) -> str:
    """Simulates a read-only balance lookup."""
    balances = {"ACC-001": 5000.00, "ACC-002": 12500.00, "ACC-003": 750.00}
    balance = balances.get(account_id, 0.0)
    return f"Account {account_id} balance: ${balance:,.2f}"


def transfer_funds(from_account: str, to_account: str, amount: float) -> str:
    """Simulates an irreversible fund transfer (mock - no real money moves)."""
    return f"Successfully transferred ${amount:,.2f} from {from_account} to {to_account}."


# Dispatch map used by the HITL loop
TOOL_IMPLEMENTATIONS = {
    "get_account_balance": lambda args: get_account_balance(**args),
    "transfer_funds":      lambda args: transfer_funds(**args),
}

print(f"Tools defined: {get_balance_tool.name}, {transfer_tool.name}")
print(f"Approval-required tools: {APPROVAL_REQUIRED_TOOLS}")

Tools defined: get_account_balance, transfer_funds
Approval-required tools: {'transfer_funds'}


## Step 3: Create agent

Create a versioned agent with both tools attached. The instructions make clear that `transfer_funds` requires human approval - reinforcing the HITL convention at the prompt level.

In [3]:
from azure.ai.projects.models import PromptAgentDefinition

agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=model_deployment,
        instructions=(
            "You are a banking assistant. "
            "You have access to two tools: get_account_balance and transfer_funds. "
            "When a user asks to transfer funds, call transfer_funds with the appropriate arguments. "
            "Do NOT describe what you will do - just call the tool directly. "
            "The system will handle human approval for transfer_funds before executing it."
        ),
        tools=[get_balance_tool, transfer_tool],
    ),
    description="HITL demo agent - financial transactions with human approval for transfers.",
)

# Build the agent_reference body used in every Responses API call
agent_ref = {"agent_reference": {"name": agent.name, "type": "agent_reference"}}

print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

Agent created (id: payments-approval-agent:1, name: payments-approval-agent, version: 1)


## Step 4: HITL loop helper

The `run_with_hitl` function implements the core pattern:

1. Call `responses.create()` with the user message
2. Inspect `response.output` for `function_call` items
3. For each tool call:
   - If the tool name is in `APPROVAL_REQUIRED_TOOLS` → prompt the operator via `input()`
   - Otherwise → auto-execute immediately
4. Submit all tool results back via `previous_response_id` to continue the conversation
5. Repeat until no more tool calls remain

> **Production note:** In a production system, replace `input()` with a webhook, UI event, or async approval queue. The pattern here illustrates the interception point.

In [4]:
def run_with_hitl(user_message: str) -> str:
    """Run the agent with HITL interception for approval-required tools.

    Returns the final text response from the agent.
    """
    # Initial request
    response = openai_client.responses.create(
        input=[{"role": "user", "content": user_message}],
        extra_body=agent_ref,
    )

    while True:
        # Check for pending tool calls in the output
        tool_calls = [item for item in response.output if item.type == "function_call"]
        if not tool_calls:
            break  # No more tool calls - we have the final text response

        tool_outputs = []
        for call in tool_calls:
            args = json.loads(call.arguments)

            if call.name in APPROVAL_REQUIRED_TOOLS:
                # ── Human-in-the-loop branch ──────────────────────────────────
                print(f"\n{'='*60}")
                print(f"[APPROVAL REQUIRED] Tool: {call.name}")
                print("Arguments:")
                for k, v in args.items():
                    print(f"  {k}: {v}")
                print(f"{'='*60}")
                decision = input("Approve this action? (yes/no): ").strip().lower()

                if decision == "yes":
                    result = TOOL_IMPLEMENTATIONS[call.name](args)
                    print(f"[APPROVED] Executing {call.name} -> {result}")
                else:
                    result = f"Action '{call.name}' was rejected by the human operator."
                    print(f"[REJECTED] {call.name} will not execute")
            else:
                # ── Auto-execute branch ───────────────────────────────────────
                result = TOOL_IMPLEMENTATIONS[call.name](args)
                print(f"[AUTO-EXECUTE] {call.name}({args}) -> {result}")

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": result,
            })

        # Submit tool results and continue
        response = openai_client.responses.create(
            input=tool_outputs,
            previous_response_id=response.id,
            extra_body=agent_ref,
        )

    return response.output_text


print("HITL helper defined - ready to run scenarios.")

HITL helper defined - ready to run scenarios.


## Step 5: Approve scenario

The agent calls `transfer_funds`. The operator is prompted and types **`yes`** to approve the transfer.

**Expected flow:**
1. Agent returns a `function_call` for `transfer_funds`
2. HITL loop prints the approval prompt
3. Operator enters `yes`
4. Tool executes and the result is submitted back
5. Agent confirms the transfer in its final response

> When the prompt appears, type **`yes`** and press Enter.

In [5]:
print("Scenario: Approve a fund transfer")
print("-" * 60)

result = run_with_hitl("Please transfer $500 from ACC-001 to ACC-002.")

print("\n" + "=" * 60)
print("Agent response:")
print(result)

Scenario: Approve a fund transfer
------------------------------------------------------------

[APPROVAL REQUIRED] Tool: transfer_funds
Arguments:
  from_account: ACC-001
  to_account: ACC-002
  amount: 500
[REJECTED] transfer_funds will not execute

Agent response:
The transfer of $500 from ACC-001 to ACC-002 was not approved. Is there anything else you would like to do?


## Step 6: Reject scenario

The agent calls `transfer_funds` again. This time the operator types **`no`** to reject the transfer.

**Expected flow:**
1. Agent returns a `function_call` for `transfer_funds`
2. HITL loop prints the approval prompt
3. Operator enters `no`
4. A rejection message is submitted as the tool result
5. Agent acknowledges the rejection in its final response

> When the prompt appears, type **`no`** and press Enter.

In [6]:
print("Scenario: Reject a fund transfer")
print("-" * 60)

result = run_with_hitl("Transfer $10,000 from ACC-002 to ACC-003.")

print("\n" + "=" * 60)
print("Agent response:")
print(result)

Scenario: Reject a fund transfer
------------------------------------------------------------

[APPROVAL REQUIRED] Tool: transfer_funds
Arguments:
  from_account: ACC-002
  to_account: ACC-003
  amount: 10000
[APPROVED] Executing transfer_funds -> Successfully transferred $10,000.00 from ACC-002 to ACC-003.

Agent response:
The transfer of $10,000 from ACC-002 to ACC-003 has been successfully completed. Is there anything else you would like to do?


## Step 7: Multi-step scenario

A single request triggers **two tool calls in sequence**: a balance check followed by a transfer.

**Expected flow:**
1. Agent calls `get_account_balance` - auto-executed immediately (no prompt)
2. Agent calls `transfer_funds` - HITL prompt appears
3. Operator approves or rejects the transfer
4. Agent provides a final summary response

This demonstrates that auto-execute and approval-required tools coexist cleanly in the same loop.

> When the prompt appears, type **`yes`** or **`no`** and press Enter.

In [7]:
print("Scenario: Multi-step - balance check (auto) then transfer (approval)")
print("-" * 60)

result = run_with_hitl(
    "Check the balance on ACC-001, then transfer $200 from ACC-001 to ACC-003."
)

print("\n" + "=" * 60)
print("Agent response:")
print(result)

Scenario: Multi-step - balance check (auto) then transfer (approval)
------------------------------------------------------------
[AUTO-EXECUTE] get_account_balance({'account_id': 'ACC-001'}) -> Account ACC-001 balance: $5,000.00

[APPROVAL REQUIRED] Tool: transfer_funds
Arguments:
  from_account: ACC-001
  to_account: ACC-003
  amount: 200
[APPROVED] Executing transfer_funds -> Successfully transferred $200.00 from ACC-001 to ACC-003.

Agent response:
The balance on account ACC-001 is $5,000. I have transferred $200 from ACC-001 to ACC-003.


## Summary

### Key concepts

| Concept | Description |
|---------|-------------|
| `APPROVAL_REQUIRED_TOOLS` | A set of tool names that trigger the human approval branch |
| `function_call` output item | The Responses API returns tool calls as output items - it does **not** execute them |
| `previous_response_id` | Continues the response loop when submitting tool results back to the agent |
| `function_call_output` | The message type used to submit tool execution results to the Responses API |
| `input()` for approval | Works in Jupyter notebooks; replace with webhook/UI event in production |

### The HITL loop

```
responses.create(user_message)
    |
    v
response.output contains function_call items?
    |
    +-- No  --> return response.output_text
    |
    +-- Yes --> for each tool_call:
                    if name in APPROVAL_REQUIRED_TOOLS:
                        prompt operator --> approve/reject
                    else:
                        auto-execute
                responses.create(tool_outputs, previous_response_id=response.id)
                loop back ^
```

### Related patterns

Three different "approval" layers exist across the stack. This notebook uses the third one because the first two do not cover its scenario (custom `FunctionTool` on a Foundry-hosted versioned agent).

**1. MAF `@tool(approval_mode="always_require")`** - client-side runtime approvals

Released in `agent-framework-core` and available in the version this repo pins (`1.0.0rc6`). Applies when the agent runs in the local MAF runtime (`Agent(client=OpenAIChatClient(), ...)`), **not** when calling a Foundry-hosted agent through the Responses API. The MAF runtime returns `user_input_requests` on the run result; the caller replies with `Message(role="user", contents=[req.create_response(True|False)])` to resume.

```python
from typing import Annotated
from agent_framework import tool

@tool(approval_mode="always_require")
def transfer_funds(from_account: Annotated[str, "Source account ID"],
                   to_account:   Annotated[str, "Destination account ID"],
                   amount:       Annotated[float, "Amount in USD"]) -> str:
    return f"Transferred ${amount:,.2f} from {from_account} to {to_account}."
```

See: [Using function tools with human in the loop approvals](https://learn.microsoft.com/en-us/agent-framework/tutorials/agents/function-tools-approvals?pivots=programming-language-python).

**2. Foundry `MCPTool(require_approval="always")`** - server-side MCP approvals

Native server-side approval routing on the Foundry Agent Service, but only for MCP tools. When approval is required the Responses API returns an `mcp_approval_request` output item; the client submits an `mcp_approval_response` to continue. See: [Connect to MCP Server Endpoints for agents](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/model-context-protocol).

**3. Manual interception** - what this notebook does

For custom `FunctionTool` definitions on a Foundry-hosted agent there is no first-class `require_approval` flag in the current SDK (`azure-ai-projects` rc6 `FunctionTool` exposes only `name` / `description` / `parameters` / `strict` / `type`). The `APPROVAL_REQUIRED_TOOLS` set + Responses API loop shown above is the canonical pattern for this scenario.

### Cleanup reminder

Run the cleanup cell below to delete the versioned agent when you are done with this lab.

In [8]:
# Cleanup - delete the versioned agent
project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print(f"Deleted agent version {agent.version} of '{agent.name}'.")

Deleted agent version 1 of 'payments-approval-agent'.
